# Gaussian Naive Bayes from Scratch

By the end of this session, the aim is to understand and implement this pipeline:

$$training~data\rarr class~priors\rarr Gasusian~feature~models\rarr Likelihoods\rarr posterior~scores\rarr predicted~class$$

## 1. Retrieval

1. Suppose $C$ is a class and $\mathbf{x}$ is the observed feature vector. Write Bayes’ theorem for: $$ P(C\mid\mathbf{x}) $$

> $ P(C\mid\mathbf{x}) = \frac{P(C)P(\mathbf{x}\mid C)}{P(\mathbf{x})}$

2. In a classification problem, what would the prior $P(C)$ represent?

> $P(C)$ is the probability that an observation belongs to class $C$ before looking at its **feature values**

3. If one feature within a class is modelled by a normal distribution, which two parameters describe that distribution?

> $\mu$ and $\sigma^2$

4. What is the difference between a prior, likelihood and posterior

> - Prior: probability that the class is $C$ before seeing the features.
> - Likelihood: how plausible the observed features are assuming the class is $C$
> - Posterior: probability that the class is $C$ after observing the features

5. If two classes are equally plausible based on their feature values, but one class is much more common in the training population, which class would you expect Bayes to favour, and why?

> The class that is more common in the training population, since this class has a higher prior meaning the same likelihood will be being applied to a higher starting probability resulting in a higher posterior

## 2. Probability density

Suppose we look at petal length among flowers belonging to one class, $C$. Gaussian Naive Bayes makes a modelling assumption:

$$X\mid C\sim\mathcal N(\mu_C,\sigma_C^2) $$

In words:

    Within class $C$, we model this feature as following a normal distribution with a class-specific mean and variance.

Imagine we learned: $$u_C=5.0 \qquad\sigma_C=0.5$$ and a new flower has: $$x=5.1$$

We want some way of answering:

    Is $5.1$ a value that looks plausible for flowers from class $C$?

For a discrete variable, we could calculate a probability such as: $$P(X=4)$$

But for a continuous measurement there are infinitely many possible exact values. Under a continuous distribution: $$P(X=5.1)=0$$

That sounds strange initially, but remember that actual probabilities come from **intervals**: $$P(5.0 < X < 5.2)$$ 

which is the area under the normal curve between those points.

The Gaussian formula instead gives us the height of the curver at a particukar value:

$$p(x\mid C)=\frac{1}{\sqrt{2\pi\sigma_C^2}}\exp\left(-\frac{(x-\mu_C)^2}{2\sigma_C^2}\right)$$

That height is called a **probability density**.

You do **not** need to memorise that equation. What matters right not is what its pieces do:
- $(x-\mu_C)^2$ measures how far the observed value is from the class mean.
- dividing by $\sigma_C^2$ judges that distance relative to the class's spread.
- therefore values near the class mean thend to have **high density**;
- values far into the tails tend to have **low density**.

So in ML terms: $$p(x\mid C)$$ answers *how compatible is this observed feature value with the distribution we've learned for class $C$?*


## 3. Calculate Gaussian density

Use the Gaussian density:

$$p(x\mid C)=\frac{1}{\sqrt{2\pi\sigma_C^2}}\exp\left(-\frac{(x-\mu_C)^2}{2\sigma_C^2}\right)$$

Implement:
```
def gaussian_density(x, mean, variance):
    ...
```

Then calculate the density of $x=4.4$ under: $$C_1:\qquad \mu=1.5,\qquad \sigma^2=0.2^2$$ and: $$C_2:\qquad \mu=4.5,\qquad \sigma^2=0.5^2$$

Print both densities and answer:

    Which class does this single feature provide stronger evidence for?

In [1428]:
import math

def gaussian_density(x, mean, variance):
    """
    Calculates the probability density for feature x given class mean and variance

    Args:
        x (float): feature value
        mean (float): class mean
        variance (float): class variance
    Returns:
        density (float): probability density for feature x 
    """

    density = (1 / math.sqrt(2 * math.pi * variance)) * math.exp(
      -((x - mean)**2) / (2 * variance)
      )

    return density

In [1429]:
mean_1, variance_1 = 1.5, 0.2**2
mean_2, variance_2 = 4.5, 0.5**2

x = 4.4

print('Probability density for x in Class 1:', gaussian_density(x, mean_1, variance_1))
print('Probability density for x in Class 2:', gaussian_density(x, mean_2, variance_2))

Probability density for x in Class 1: 4.412377487297412e-46
Probability density for x in Class 2: 0.782085387950912


> This single feature provides stronger evidence for class 2, as the probability density in class 2 is approximatley 0.78, compared to 4.41e-46 for class 2

## 4. New concept: Likelihood

There are two different stages in what we're building. At prediction time, we know the Gaussian parameters for a class and ask:

> How compatible is this new feature value with that fitted distribution?

That's what you just calculated: $$p(x\mid C,\mu_C,\sigma_C^2)$$

Here the parameters are fixed and $x$ varies.

At training time, however, we have the opposite problem.

Imagine the training observaions for one feature in one class are: $$x1,x2,...,x_n$$

We know those measurements because they're our training data, but we don't yet know what Gaussian should represent them.

So now the data are fixed and we ask:

> Which values of $\mu$ and $\sigma^2$ would make the observed training data most plausible?

The same Gaussian density formula is involved, but we are now treating it as a **function of the parameters** rather than the observation. The function is called the **likelihood**.

For one observation: $$L(\mu,\sigma^2\mid x_1,\dots,x_n)=\prod_{i=1}^n p(x_i\mid\mu,\sigma^2)$$

Why multiply? Because we're asking for parameter values that make **all of the observed training measurements jointly plausible**.

### Maximum likelihood estimation

We now need to choose the Gaussian parameters.

**Maximum likelihood estimation (MLE)** means:

> Choose the parameter values that maximise the likelihood of the training data we actually observed.

Symbolically: $$\hat{\mu},\hat{\sigma}^2=\arg\max_{\mu,\sigma^2}L(\mu,\sigma^2)$$

The $\arg\max$ notation means:

> Return the **parameter values** at which the function reaches its maximum.

For a Gaussian, carrying out that optimisation gives a very familiar result:

$$\hat{\mu}=\frac{1}{n}\sum_{i=1}^nx_i$$

and: $$\hat{\sigma}^2=\frac{1}{n}\sum_{i=1}^n(x_i-\hat{\mu})^2$$

So there's an important ML interpretation hiding behind something you've done many times:

> When Gaussian Naive Bayes calculates the mean and variance of a feature within a class, it is **fitting the Gaussian distribution to the training data using maximum likelihood**.


## 5. From MLE to training Gaussian Naive Bayes

For Gaussian data, solving that optimisation gives: $$\hat\mu = \frac{1}{n}\sum_{i=1}^nx_i$$

and: $$\hat{\sigma}^2=\frac{1}{n}\sum_{i=1}^n(x_i-\hat{\mu})^2$$

Notices that MLE avriance uses $n$, not $n-1$.

Youv'e probably encountered sample variance using: $$\frac{1}{n-1}\sum_{i=1}^n(x_i-\bar{x})^2$$

The distinction is:
- $\frac{1}{n-1}$ gives the common **unbiased estimator** of population variance
- $\frac{1}{n}$ is the variance that **maximises the Gaussian likelihod**

So for our implementation:
```python
np.var(values, ddof=0)
```
is what we want.

We don't need to derive the $n$ versus $n-1$ result today; just understand why we're deliberately chossing `ddof=0`.

### Training data
Now create the notebook's actual dataset:

In [1430]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import numpy as np

iris = load_iris()

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Before doing anything else, inspect:

In [1431]:
print(X.shape)
print(y.shape)
print(X_train.shape)
print(X_test.shape)
print(np.unique(y, return_counts=True))
print(np.unique(y_train, return_counts=True))

(150, 4)
(150,)
(120, 4)
(30, 4)
(array([0, 1, 2]), array([50, 50, 50]))
(array([0, 1, 2]), array([40, 40, 40]))


Think about what each row and column of `X` represents.

### Your training task
Gaussian Naive Bayes needs to learn **three things** from `X_train` and `y_train`:

For each class $C_k$: $$P(C_k)$$

and, for every feature $j$: $$\mu_{kj}$$ $$\sigma_{kj}^2$$

With Iris there are:
- 3 classes
- 4 features

So before coding, predict the required array shapes for:
```python
priors:
means:
variances:
```

> - priors: (3,)
> - means: (3, 4)
> - variances: (3, 4)

Then implement the calculations.

A useful way to approach it is to iterate over the unique classes and extract the training observations belonging to that class:
```python
classes = np.unique(y_train)

for c in classes:
    X_c = ...
```

From `X_c`, think carefully about which axis you need for the mean and variance if you want one value per feature.

Don't build prediction yet. For now our only goal is to train the probability model and end up with correctly shaped **priors, means and variances**.

In [1432]:
classes = np.unique(y_train)

priors = []
means = []
variances = []

for c in classes:
    X_c = X_train[y_train == c]

    prior = len(X_c) / len(X_train)
    mean = np.mean(X_c, axis=0)
    variance = np.var(X_c, ddof=0, axis=0)

    priors.append(prior)
    means.append(mean)
    variances.append(variance)

priors = np.array(priors)
means = np.array(means)
variances = np.array(variances)

print(f'Priors shape: {priors.shape},\nPriors:\n{priors}\n')
print(f'Means shape: {means.shape},\nMeans:\n{means}\n')
print(f'Variances shape: {variances.shape},\nVariances:\n{variances}')

Priors shape: (3,),
Priors:
[0.33333333 0.33333333 0.33333333]

Means shape: (3, 4),
Means:
[[4.985  3.415  1.4775 0.255 ]
 [5.93   2.75   4.2525 1.32  ]
 [6.61   2.98   5.58   2.04  ]]

Variances shape: (3, 4),
Variances:
[[0.092775   0.155275   0.02524375 0.012975  ]
 [0.2216     0.093      0.19149375 0.0341    ]
 [0.4574     0.1221     0.3236     0.0704    ]]


## 6. New concept: the "naive" assumption

Suppose a new observation has four features: $$\mathbf{x}=[x_1,x_2,x_3,x_4]$$

What we really want for each class i: $$p(\mathbf{x}\mid C)$$

That means:

> how plausible is this entire combination of four feature values if the observation belongs to class $C_k$?

In general, modelling that full joint distribution can be difficult because the features may depend on each other.

Naive Bayes makes a simplifying assumption:

> The features are conditionally independent given the class.

So once we know $C_k$, it assumes that knowing one feature gives us no additional information about another feature.

Mathematically, this lets us replace the joint density with: $$p(\mathbf{x}\mid C_k)=\prod_{j=1}^4p(x_j\mid C_k)$$

> Explanation:
>
> Without independence, the full joint density can be expanded as: 
> $$p(x_1,x_2,x_3,x_4\mid C_k)\\=p(x_1\mid C_k)p(x_2\mid x_1,C_k)p(x_3\mid x_1,x_2,C_k)p(x_4\mid x_1,x_2,x_3,C_k)$$
> 
> This is hard to model because we need distributions describing how features behave **in combination with others**.
>
> The Naive Bayes assumption says that, given $C$: $$p(x_2\mid x_1,C)=p(x_2\mid C)$$ and similarly:
> $$p(x_3\mid x_1,x_2,C_k)=p(x_3\mid C)\\ p(x_4\mid x_1,x_2,x_3,C_k)=p(x_4\mid C)$$
>
> Therefore the whole thing collapses to: $$p(\mathbf{x}\mid C_k)=p(x_1\mid C_k)p(x_2\mid C_k)p(x_3\mid C_k)p(x_4\mid C_k)$$
>
> or compactly: $$p(\mathbf{x}\mid C_k)=\prod_{j}p(x_j\mid C_k)$$


So for Iris: $$p(\mathbf{x}\mid C_k)=p(x_1\mid C_k)p(x_2\mid C_k)p(x_3\mid C_k)p(x_4\mid C_k)$$

Each individual term is something your `gaussian_density()` function can calculate.

This assumption is why it is called **naive**. Iris features such as petal length and petal width are actually correlated, so the assumption ins't literally true. But the simplification can still give a useful classifier.

And this connects directly to Bayes: $$P(C_k\mid\mathbf{x})=\frac{P(C_k)P(\mathbf{x}\mid C_k)}{P(\mathbf{x})}$$

Substituting the naive assumption: $$P(C_k\mid\mathbf{x})=\frac{P(C_k)\prod_{j}p(x_j\mid C_k)}{P(\mathbf{x})}$$

Now here's the useful classification trick:

When we're deciding which class wins, $p(\mathbf{x})$ is the **same denominator for every class**. So we don't actually need to calculate it. 

Therefore, we can compare: $$\boxed{P(C_k)\prod_{j}p(x_j\mid C_k)}$$

for each class and choose the largest.

That is an **unnormalised posterior score**.

### Quick understanding check

Before we code this, answer these two:

1. Why is Gaussian Naive Bayes allowed to multiply the four individual feature densities together?

> Gaussian Naive Bayes makes an assumption that the features are conditionally independent given the class. This allows it to factor the joint density: $$p(\mathbf{x}\mid C_k)=\prod_jp(x_j\mid C_k)$$

2. Why can we ignore $p(\mathbf{x})$ when all we want is the predicted class?

> The Bayes' calculation for each classes posterior will contain the same denominator, $P(\mathbf{x})$, therefore if we just need to know which class has the highest posterior we can ignore this term when we compare the results

## 7. Classify one real test observation

Take:
```python
X_test[0]
```
This has shape:
```python
(4,)
```
because it's one flower with four features.

For each class $C_k$, we now need:
1. the density of each of the four observed feature values under that class;
2. the product of those densities;
3. multiply that class's prior.

So: $$score_k=P(C_k)\prod_{k=1}^{4}p(x_j\mid C_k)$$

Your final `scores` array should have shape:
```python
(3,)
```
$\rarr$ one score for each class.

### Coding task
Use your existing `gaussian_density()` function. Since it currently uses Python's `math` module and accepts scalar values, I'd keep this first implementation explicit rather than vectorising it yet.

In [1433]:
x = X_test[0]

scores = []

for class_index, c in enumerate(classes):
  joint_density = 1
  for feature_index in range(len(x)):
    joint_density *= gaussian_density(
      x[feature_index], 
      means[class_index, feature_index], 
      variances[class_index, feature_index]
      )

  score = priors[class_index] * joint_density
  scores.append(score)

print('Observations:', x)
print('Class scores:', scores)
print('Predicted class:', classes[scores.index(max(scores))])

Observations: [4.4 3.  1.3 0.2]
Class scores: [np.float64(0.1683150204465115), np.float64(3.5417380604433776e-21), np.float64(2.0878892838007087e-26)]
Predicted class: 0


## 8. Why Naive Bayes uses logarithms

Real datasets may have tens, hundreds, or thousands of features. Multiplying many numbers smaller than $1$ to calculate the joint density can eventually produce a value so tiny that floating-point arithmetic represents it as exactly $0$.

This is called **numerical underflow**.

We therefore ususally work with the logarithm of the score.

starting with: $$score_k=P(C_k)\prod_{k=1}p(x_j\mid C_k)$$

take the logarithm: $$\log(score_k)=\log P(C_k)+\log\left(\prod_jp(x_i\mid C_k)\right)$$

and because: $$\log(ab)=\log a + \log b$$

we get: $$\boxed{\log(score_k)=\log P(C_k)+\sum_j\log p(x_j\mid C_k)}$$

So instead of multiplying many tiny desnities, **we add their logarithms**.

Crucially, this does not change which class wins because $\log(x)$ is strictly increasing: 

$$a>b \quad\Longrightarrow\quad \log x>\log b$$

Therefore: $$\arg\max_k\text{score}_k = \arg\max_k\log(\text{score}_k)$$

### Next task
Modify your one-observation classifier so that it calculates log scores rather than multiplying densities into a joint density.

The output should be:
- the three log scores;
- the predicted class label;
- a check against the true label for X_test[0].

In [1434]:
log_scores = []

for class_index, c in enumerate(classes):
  sum_log_densities = 0
  for feature_index in range(len(x)):
    sum_log_densities += math.log(
      gaussian_density(
        x[feature_index], 
        means[class_index, feature_index], 
        variances[class_index, feature_index]
        )
      )

  log_score = math.log(priors[class_index]) + sum_log_densities
  log_scores.append(log_score)

print('Class log(scores):', log_scores)
print('Predicted class:', classes[log_scores.index(max(log_scores))])
print('Log scores class matches true class:', classes[log_scores.index(max(log_scores))]==y_test[0])

Class log(scores): [-1.78191793371955, -47.089669368692924, -59.131058774230326]
Predicted class: 0
Log scores class matches true class: True


## 9. Direct Gaussian log-density

When calculating one individual Gaussian density, the Gaussian contains: $$\exp \left (-\frac{(x-\mu)^2}{2\sigma^2}\right )$$

If $x$ is extremely far from $\mu$, that exponential can be so tiny that `gaussian_density()` returns 0.0 due to numerical underflow. Therefore, even if we run:
```python
math.log(gaussian_density(...))
```
the underflow has already happened inside of `gaussian_density()`.

Therefore the best practise is to calculate the Gaussian log-density directly because this prevents **individual-density underflow**.

$$\log p(x\mid C) = -\frac{1}{2}\log(2\pi\sigma^2)-\frac{(x-\mu)^2}{2\sigma^2}$$

### Task
1. Write a function:
```python
gaussian_log_density(x, mean, variancE)
```
that implements the log formula directly.
2. Update your one-observation classifier so that it uses this function rather than:
```python
math.log(gaussian_density(...))
```
3. For `X_test[0]`, output:
    - the three log scores;
    - the predicted class label;
    - the true class label `y_test[0]`;
    - whether the prediction is correct.
4. Compare the new log scores with the ones from your previuous implementation. They should be essentially the same for this observation. 

In [1435]:
def gaussian_log_density(x, mean, variance):
    """
    Calculates the log probability density for feature x given class mean and variance

    Args:
        x (float): feature value
        mean (float): class mean
        variance (float): class variance
    Returns:
        log_density (float): log probability density for feature x 
    """

    log_density = - (1 / 2) * math.log(2 * math.pi * variance) - ((x - mean)**2 / (2 * variance))

    return log_density

In [1436]:
new_log_scores = []

for class_index, c in enumerate(classes):
  sum_new_log_densities = 0
  for feature_index in range(len(x)):
    sum_new_log_densities += gaussian_log_density(
      x[feature_index], 
      means[class_index, feature_index], 
      variances[class_index, feature_index]
      )

  new_log_score = math.log(priors[class_index]) + sum_new_log_densities
  new_log_scores.append(new_log_score)

print('Class log(scores):', new_log_scores)
print('Predicted class:', classes[new_log_scores.index(max(new_log_scores))])
print('Log scores class matches true class:', classes[new_log_scores.index(max(new_log_scores))]==y_test[0])

Class log(scores): [np.float64(-1.7819179337195508), np.float64(-47.089669368692924), np.float64(-59.131058774230326)]
Predicted class: 0
Log scores class matches true class: True


Why is calculating the Gaussian log-density directly more numerically stable than first calculating gaussian_density() and then taking its logarithm?

> The gaussian_density function may result in a density value of 0.0 due to numerical underflow occuring in the instance where $x$ is extremley far from $\mu$ relatinve to $\sigma$. Even if we take the log of this gaussian_density() result, the underflow has already occured therefore information has been lost. The direct log density calcualation never evaluates the exponential which means the tiny intermediate value that could underflow is never created. 

## 10. Extend the classifier to the full test set

You have now built the complete calculation required to predict one observation. Your next task is to use it to predict every observation in `X_test`.

For each observation: $$\mathbf{x}^{(i)}$$

calculate the log score for every class: $$\log(score_k)=\log P(C_k)+\sum_j\log p(x_j\mid C_k)$$

and choose the class with the largest score.

Store all predictions in an array called: `y_pred`

Expected shape: `(30,)`

Then calcualte **accuracy**, which we'll define here as simply:

$$\text{accuracy}=\frac{\text{number of correct predictions}}{\text{total number of predictions}}$$

Output:
- y_pred
- y_test
- the number of correct predictions
- the accuracy

In [1437]:
y_pred=[]
log_scores = []

for x in X_test:
  log_scores_row = [] 
  for class_index, c in enumerate(classes):
    sum_log_densities = 0
    for feature_index in range(len(x)):
      sum_log_densities += gaussian_log_density(
        x[feature_index],
        means[class_index, feature_index],
        variances[class_index, feature_index]
      )

    log_score = math.log(priors[class_index]) + sum_log_densities
    log_scores_row.append(log_score)

  predicted_class = classes[log_scores_row.index(max(log_scores_row))]
  y_pred.append(predicted_class)
  log_scores.append(log_scores_row)

y_pred = np.array(y_pred)
n_correct_predictions = np.count_nonzero(y_pred == y_test)
accuracy = np.count_nonzero(y_pred == y_test) / len(y_pred)

print('Predicted classes:', y_pred)
print('Actual classes:   ', y_test)
print('Number of correct predictions:', n_correct_predictions)
print('Model accuracy:', accuracy)


Predicted classes: [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 2 1 0 2 0]
Actual classes:    [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 1 1 0 2 0]
Number of correct predictions: 29
Model accuracy: 0.9666666666666667


## 11. Compare with scikit-learn

You've built the model yourself, so now we can use scikit-learn without it becoming a black box.

Use: `sklearn.naive_bayes.GaussianNB`

Using exactly the same `X_train`, `X_test`, `y_train`, and `y_test`:
1. Train a `GaussianNB` model.
2. Predict `X_test`.
3. Calculate the test accuracy.
4. Compare its predictions against your `y_pred`.
5. Inspect the fitted model attributes corresponding to:
    - class priors;
    - per-class feature means;
    - per-class feature variances.

In [1438]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()

gnb.fit(X_train, y_train)

y_pred_sklearn = gnb.predict(X_test)

n_correct_predictions_sklearn = np.count_nonzero(y_pred_sklearn == y_test)
accuracy_sklearn = n_correct_predictions_sklearn / len(y_pred_sklearn)
print('sklearn model accuracy:', accuracy_sklearn)

model_agreement = y_pred == y_pred_sklearn
print(model_agreement)

print('\nsklearn model priors:', gnb.class_prior_)
print('\nsklearn model means:\n', gnb.theta_)
print('\nsklearn model variances:\n', gnb.var_)


sklearn model accuracy: 0.9666666666666667
[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True]

sklearn model priors: [0.33333333 0.33333333 0.33333333]

sklearn model means:
 [[4.985  3.415  1.4775 0.255 ]
 [5.93   2.75   4.2525 1.32  ]
 [6.61   2.98   5.58   2.04  ]]

sklearn model variances:
 [[0.092775   0.155275   0.02524375 0.012975  ]
 [0.2216     0.093      0.19149375 0.0341    ]
 [0.4574     0.1221     0.3236     0.0704    ]]


### Manual model:
Priors:
[0.33333333 0.33333333 0.33333333]

Means:
[[4.985  3.415  1.4775 0.255 ]
 [5.93   2.75   4.2525 1.32  ]
 [6.61   2.98   5.58   2.04  ]]

Variances:
[[0.092775   0.155275   0.02524375 0.012975  ]
 [0.2216     0.093      0.19149375 0.0341    ]
 [0.4574     0.1221     0.3236     0.0704    ]]

### sklearn:
Priors: [0.33333333 0.33333333 0.33333333]

Means:
 [[4.985  3.415  1.4775 0.255 ]
 [5.93   2.75   4.2525 1.32  ]
 [6.61   2.98   5.58   2.04  ]]

Variances:
 [[0.092775   0.155275   0.02524375 0.012975  ]
 [0.2216     0.093      0.19149375 0.0341    ]
 [0.4574     0.1221     0.3236     0.0704    ]]


### Answer:

> Do scikit-learn's learned parameters match the parameters we estimated manually?

> The scikit-learn priors and means match the manually estimated parameters, while the variances are numerically equivalent apart from scikit-learn's very small variance-smoothing adjustment.

and:

> Does scikit-learn make exactly the same 30 predictions as our implementation?

> Yes, it incorrectly predicted the same single test observation

# 12. Error analysis

Find the single misclassified observation and report:
- its position within `X_test`;
- its four feature values;
- its true class;
- its predicted class;
- the three class log scores for that observation.

In [1439]:
incorrect_index = np.argwhere(y_pred != y_test)
print('Index of model error:', incorrect_index[0,0])
print('Feature values:\n',X_test[incorrect_index[0,0]])
print(f'True class: {y_test[incorrect_index[0,0]]}, Predicted class: {y_pred[incorrect_index[0,0]]}')
print('Class log scores:\n',log_scores[incorrect_index[0,0]])

Index of model error: 25
Feature values:
 [6.7 3.  5.  1.7]
True class: 1, Predicted class: 2
Class log scores:
 [np.float64(-341.2756310511818), np.float64(-5.567704400179849), np.float64(-2.79219854545347)]


> The model strongly favoured class 2 over the true class 1. The class-2 score was approximately 16 times the class-1 score, while class 0 was effectively ruled out. This suggests that the observed feature combination looks substantially more like the Gaussian distributions learned for class 2 than those learned for class 1.

Now we want to understand **why** this class 1 flower looked so much like class 2/ For the misclassified observation: $$[6.7, 3.0, 5.0, 1.7]$$

calculate the individual Gaussian log-density contributions from each of the four features for:
- class 1
- class 2

Report the two sets of four values, then answer:

> Which feature or features most strongly push the model toward class 2 rather than the true class 1?

In [1440]:
x_error_analysis = X_test[incorrect_index[0,0]]
log_densities = {}
check_classes = [1, 2]

for class_index, c in enumerate(classes):
  if c not in check_classes: continue

  log_density_row = []
  for feature_index, feature in enumerate(x_error_analysis):
    log_density = gaussian_log_density(feature, means[class_index, feature_index], variances[class_index, feature_index])
    log_density_row.append(log_density)

  log_densities[c] = log_density_row
  print(f'Class: {c}, log-density: {log_density_row}')

Class: 1, log-density: [np.float64(-1.5032686292729196), np.float64(-0.06738214566657502), np.float64(-1.5514296979279791), np.float64(-1.3470116386442657)]
Class: 2, log-density: [np.float64(-0.5366944289110932), np.float64(0.13088091409006475), np.float64(-0.8745924893960074), np.float64(-0.413180252568324)]


> All for features favour class 2 over the true class 1. Features 0 and 3 provide the strongest relative evidence for class 2, followed by feature 2, while feature 1 contributes comparatively little to the incorrect decision.

## 13. Test the "naive" assumption

Now we can connect back to the asumption at the heart of Naive Bayes:

> Features are conditionally independent given the class.

A useful way to look for violations of that assumption is correlation.

You already know **covariance** from PCA. Correlation is essentially covariance scaled by the standard deviation of the two variables:

$$ \rho_{XY} = \frac{\operatorname{Cov}(X,Y)} {\sigma_X\sigma_Y} $$

Unlike covariance, correlation is unitless and lies between: $$-1\geq\rho\leq 1$$

- near $+1$: strong positive linear relationship
- near $-1$: strong negative linear relationship
- near $0$: little linear relationship

One important qualification: **independence implies zero correlation in ordinary finite-variance settings, but zero correlation does not generally prove independence**. So strong within-class correlation is evidence that the Naive Bayes assumption is violated; weak correlation doesn't prove the assumption is true. 

### Task

For **each class separatley**, use only its observations from `X_train` and calculate the $4\times4$ feature correlation matrix.

This must be **within each class**, rather than across the whole dataset, because Naive Bayes assumes independence **conditional on knowing the class**.

For each class report:
1. its correlation matrix;
2. the pair of different features with the largest absolute correlation;
3. that correlation value.

In [1441]:
# for class_index in classes:
X_train_dict = {}

for index, observation in enumerate(X_train):
  c = y_train[index]
  if c not in X_train_dict: 
    X_train_dict[c] = []

  X_train_dict[c].append(observation)
  
for c in classes:
  print('\nClass:', c)
  observations_array = np.array(X_train_dict[c])
  cov_matrix = np.cov(observations_array, rowvar=False)

  correlation_matrix = np.corrcoef(cov_matrix, rowvar=False)
  print('Correlation matrix:\n', correlation_matrix)

  highest_correlation = 0
  highest_feature_pair = []

  for i in range(len(correlation_matrix[0,:])):
    for j in range(i + 1):
      if i == j: continue

      if abs(correlation_matrix[i, j]) > abs(highest_correlation):
        highest_correlation = correlation_matrix[i, j]
        highest_feature_pair = [i, j]

  print('Feature pair with largest absolute correlation:', highest_feature_pair)
  print('Highest correlation:', highest_correlation)


Class: 0
Correlation matrix:
 [[ 1.          0.9290782  -0.25155446  0.41584955]
 [ 0.9290782   1.         -0.09130998  0.3029233 ]
 [-0.25155446 -0.09130998  1.         -0.97377046]
 [ 0.41584955  0.3029233  -0.97377046  1.        ]]
Feature pair with largest absolute correlation: [3, 2]
Highest correlation: -0.9737704603345974

Class: 1
Correlation matrix:
 [[ 1.          0.12420937  0.77974448  0.4590633 ]
 [ 0.12420937  1.         -0.00859116 -0.08139036]
 [ 0.77974448 -0.00859116  1.          0.91413347]
 [ 0.4590633  -0.08139036  0.91413347  1.        ]]
Feature pair with largest absolute correlation: [3, 2]
Highest correlation: 0.9141334650446242

Class: 2
Correlation matrix:
 [[ 1.          0.43845044  0.9802331  -0.65707759]
 [ 0.43845044  1.          0.36388569 -0.95606879]
 [ 0.9802331   0.36388569  1.         -0.56644378]
 [-0.65707759 -0.95606879 -0.56644378  1.        ]]
Feature pair with largest absolute correlation: [2, 0]
Highest correlation: 0.9802330959272189


Answer:

> Based on these results, does the conditional-independence assumption appear realistic for the Iris features?

> The conditional-independence assumption does not appear realistic, since for all three classes the highest absolute correlation between features was over 0.9, suggetsing that these features are not conditionally independent given the class.

Naive Bayes deliberately makes an unrealistic simplifying assumption. That assumption lets the difficult joint density factorise into separate one-feature densities. Real features can be dependent even given the class, yet the classifier can still work well. Correlated features can cause related evidence to be counted more than once.